# Notebook 5: Grover's Search and Shor's Period-Finding Algorithms

This notebook is part of the FDP hands-on session on **Grover's and Shor's Algorithms**. We cover:
1. **Grover's Search Algorithm**: Implementing the oracle for target state $|101\rangle$, the Grover diffuser, and measuring the target with high probability.
2. **Quantum Fourier Transform (QFT)**: Building the forward and inverse QFT circuits.
3. **Shor's Period-Finding**: Setting up a phase estimation circuit to find the period of $a = 7 \pmod{15}$.

We will implement these models in both **Qiskit** and **PennyLane**.

## Section 1: Grover's Search Algorithm (n=3, Target=|101|)

Grover's algorithm searches an unstructured database of size $N = 2^n$ in $\mathcal{O}(\sqrt{N})$ queries. It consists of:
1. **Superposition**: Initialize all qubits in $|+\rangle$.
2. **Oracle**: Flips the phase of the target state $|101\rangle$.
3. **Diffuser**: Reflects amplitudes about the uniform superposition state $|s\rangle$.

In [1]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator
import numpy as np

# 1. Define the Oracle for target state |101>
# We apply a phase flip (-1) only to |101>
def get_grover_oracle():
    qc = QuantumCircuit(3)
    # Map |101> to |111> by flipping qubit 1
    qc.x(1)
    # Apply Multi-Controlled Z (MCZ) using Hadamard and MCX
    qc.h(2)
    qc.mcx([0, 1], 2)
    qc.h(2)
    # Map back
    qc.x(1)
    return qc

# 2. Define the Grover Diffuser (Reflection about |s>)
def get_diffuser():
    qc = QuantumCircuit(3)
    qc.h([0, 1, 2])
    qc.x([0, 1, 2])
    # Apply MCZ
    qc.h(2)
    qc.mcx([0, 1], 2)
    qc.h(2)
    qc.x([0, 1, 2])
    qc.h([0, 1, 2])
    return qc

# 3. Construct Grover Circuit (2 iterations is optimal for n=3)
qr = QuantumRegister(3)
cr = ClassicalRegister(3)
qc_grover = QuantumCircuit(qr, cr)

# Step 1: Initialize superposition
qc_grover.h([0, 1, 2])
qc_grover.barrier()

# Step 2: Iteration 1
qc_grover.compose(get_grover_oracle(), inplace=True)
qc_grover.compose(get_diffuser(), inplace=True)
qc_grover.barrier()

# Step 3: Iteration 2
qc_grover.compose(get_grover_oracle(), inplace=True)
qc_grover.compose(get_diffuser(), inplace=True)
qc_grover.barrier()

# Step 4: Measure
qc_grover.measure([0, 1, 2], [0, 1, 2])

print("Grover's Search Circuit (3 Qubits, 2 Iterations):")
print(qc_grover.draw(output='text'))

Grover's Search Circuit (3 Qubits, 2 Iterations):
      ┌───┐ ░           ┌───┐┌───┐               ┌───┐┌───┐      ░           »
q0_0: ┤ H ├─░────────■──┤ H ├┤ X ├────────────■──┤ X ├┤ H ├──────░────────■──»
      ├───┤ ░ ┌───┐  │  ├───┤├───┤┌───┐       │  ├───┤├───┤      ░ ┌───┐  │  »
q0_1: ┤ H ├─░─┤ X ├──■──┤ X ├┤ H ├┤ X ├───────■──┤ X ├┤ H ├──────░─┤ X ├──■──»
      ├───┤ ░ ├───┤┌─┴─┐├───┤├───┤├───┤┌───┐┌─┴─┐├───┤├───┤┌───┐ ░ ├───┤┌─┴─┐»
q0_2: ┤ H ├─░─┤ H ├┤ X ├┤ H ├┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├─░─┤ H ├┤ X ├»
      └───┘ ░ └───┘└───┘└───┘└───┘└───┘└───┘└───┘└───┘└───┘└───┘ ░ └───┘└───┘»
c0: 3/═══════════════════════════════════════════════════════════════════════»
                                                                             »
«      ┌───┐┌───┐               ┌───┐┌───┐      ░ ┌─┐      
«q0_0: ┤ H ├┤ X ├────────────■──┤ X ├┤ H ├──────░─┤M├──────
«      ├───┤├───┤┌───┐       │  ├───┤├───┤      ░ └╥┘┌─┐   
«q0_1: ┤ X ├┤ H ├┤ X ├───────■──┤ X ├┤ H ├──────░──╫─┤M├───

In [2]:
# Simulate the Grover search circuit after transpiling
sim = AerSimulator()
t_qc_grover = transpile(qc_grover, sim)
counts_grover = sim.run(t_qc_grover, shots=1000).result().get_counts()
print("\nMeasured outcomes (target is 101, represented in little-endian format q2 q1 q0):")
print(counts_grover)


Measured outcomes (target is 101, represented in little-endian format q2 q1 q0):
{'101': 944, '011': 13, '010': 8, '000': 9, '111': 10, '100': 10, '001': 2, '110': 4}


### Grover's Search in PennyLane

In [3]:
import pennylane as qml

dev_grover = qml.device("default.qubit", wires=3)

def oracle_pl():
    # Flip phase of |101>
    qml.FlipSign([1, 0, 1], wires=[0, 1, 2])

def diffuser_pl():
    for i in range(3):
        qml.Hadamard(wires=i)
    # Flip sign of all except |000> (or flip sign of |000>)
    qml.FlipSign([0, 0, 0], wires=[0, 1, 2])
    for i in range(3):
        qml.Hadamard(wires=i)

@qml.qnode(dev_grover)
def run_grover_pl():
    for i in range(3):
        qml.Hadamard(wires=i)
    
    # Iteration 1
    oracle_pl()
    diffuser_pl()
    
    # Iteration 2
    oracle_pl()
    diffuser_pl()
    
    return qml.probs(wires=[0, 1, 2])

probs_grover = run_grover_pl()
print("PennyLane Grover Probabilities:")
for state, prob in enumerate(probs_grover):
    print(f"State |{state:03b}>: {prob:.4f}")

PennyLane Grover Probabilities:
State |000>: 0.0078
State |001>: 0.0078
State |010>: 0.0078
State |011>: 0.0078
State |100>: 0.0078
State |101>: 0.9453
State |110>: 0.0078
State |111>: 0.0078


## Section 2: Quantum Fourier Transform (QFT)

The QFT is the quantum analogue of the discrete Fourier transform. It maps a state $|j\rangle$ to:
$$\text{QFT}|j\rangle = \frac{1}{\sqrt{N}} \sum_{k=0}^{N-1} e^{2\pi i j k / N} |k\rangle$$

We construct the forward QFT and the inverse QFT (IQFT) circuits below.

In [4]:
# 3-qubit QFT Circuit
def get_qft_3():
    qc = QuantumCircuit(3)
    # Qubit 0
    qc.h(0)
    qc.cp(np.pi/2, 1, 0)
    qc.cp(np.pi/4, 2, 0)
    # Qubit 1
    qc.h(1)
    qc.cp(np.pi/2, 2, 1)
    # Qubit 2
    qc.h(2)
    # Swaps
    qc.swap(0, 2)
    return qc

print("3-Qubit QFT Circuit Diagram:")
print(get_qft_3().draw(output='text'))

3-Qubit QFT Circuit Diagram:
     ┌───┐                                        
q_0: ┤ H ├─■────────■───────────────────────────X─
     └───┘ │P(π/2)  │       ┌───┐               │ 
q_1: ──────■────────┼───────┤ H ├─■─────────────┼─
                    │P(π/4) └───┘ │P(π/2) ┌───┐ │ 
q_2: ───────────────■─────────────■───────┤ H ├─X─
                                          └───┘   


## Section 3: Shor's Period-Finding (a = 7 mod 15)

To factor $N=15$, we choose a coprime number $a=7$ and use Quantum Phase Estimation (QPE) to find the period $r$ of $f(x) = 7^x \pmod{15}$.

The period is the smallest integer $r > 0$ such that $7^r \equiv 1 \pmod{15}$. 
Here, the powers of $7 \pmod{15}$ are: $7^0 = 1$, $7^1 = 7$, $7^2 = 4$, $7^3 = 13$, $7^4 = 1$. The period is $r = 4$.

We set up the phase estimation circuit using a 3-qubit estimation register (precision) and 4-qubit target register.

In [5]:
# We implement the controlled modular multiplication U^2^j for a=7 mod 15:
# U |y> = |7y mod 15>
# Since r=4, U^4 = I. Thus, U^2^0 = U (7 mod 15), U^2^1 = U^2 (4 mod 15), U^2^2 = U^4 = I.

def apply_controlled_u_7_15(qc, control_qubit, power):
    # Map inputs to targets:
    # For power=1 (U^1): y -> 7y mod 15. We can write a specific permuting circuit:
    if power == 1:
        # Modular multiplication by 7 mod 15
        qc.cx(control_qubit, 4) # targets are 3,4,5,6
        qc.cx(control_qubit, 5)
    elif power == 2:
        # Modular multiplication by 4 mod 15
        qc.cx(control_qubit, 3)
        qc.cx(control_qubit, 5)
    elif power == 4:
        # U^4 = Identity (no gates applied)
        pass

# Build the full QPE circuit
qc_shor = QuantumCircuit(7, 3) # 3 qubits for estimation (0, 1, 2), 4 for target (3, 4, 5, 6)

# 1. Initialize target in state |1>
qc_shor.x(3)

# 2. Put estimation qubits in superposition
qc_shor.h([0, 1, 2])
qc_shor.barrier()

# 3. Controlled-U operations
apply_controlled_u_7_15(qc_shor, 0, 1) # U^2^0
apply_controlled_u_7_15(qc_shor, 1, 2) # U^2^1
apply_controlled_u_7_15(qc_shor, 2, 4) # U^2^2
qc_shor.barrier()

# 4. Apply Inverse QFT to estimation register
# IQFT is the adjoint of QFT
iqft = get_qft_3().to_gate().inverse()
qc_shor.append(iqft, [0, 1, 2])
qc_shor.barrier()

# 5. Measure estimation register
qc_shor.measure([0, 1, 2], [0, 1, 2])

print("Shor's Period-Finding QPE Circuit:")
print(qc_shor.draw(output='text'))

Shor's Period-Finding QPE Circuit:
     ┌───┐ ░                      ░ ┌────────────────┐ ░ ┌─┐      
q_0: ┤ H ├─░───■─────────■────────░─┤0               ├─░─┤M├──────
     ├───┤ ░   │         │        ░ │                │ ░ └╥┘┌─┐   
q_1: ┤ H ├─░───┼────■────┼────■───░─┤1 circuit-54_dg ├─░──╫─┤M├───
     ├───┤ ░   │    │    │    │   ░ │                │ ░  ║ └╥┘┌─┐
q_2: ┤ H ├─░───┼────┼────┼────┼───░─┤2               ├─░──╫──╫─┤M├
     ├───┤ ░   │  ┌─┴─┐  │    │   ░ └────────────────┘ ░  ║  ║ └╥┘
q_3: ┤ X ├─░───┼──┤ X ├──┼────┼───░────────────────────░──╫──╫──╫─
     └───┘ ░ ┌─┴─┐└───┘  │    │   ░                    ░  ║  ║  ║ 
q_4: ──────░─┤ X ├───────┼────┼───░────────────────────░──╫──╫──╫─
           ░ └───┘     ┌─┴─┐┌─┴─┐ ░                    ░  ║  ║  ║ 
q_5: ──────░───────────┤ X ├┤ X ├─░────────────────────░──╫──╫──╫─
           ░           └───┘└───┘ ░                    ░  ║  ║  ║ 
q_6: ──────░──────────────────────░────────────────────░──╫──╫──╫─
           ░               

In [6]:
# Simulate the QPE circuit to find period phase after transpiling
t_qc_shor = transpile(qc_shor, sim)
counts_shor = sim.run(t_qc_shor, shots=1000).result().get_counts()
print("\nMeasured Phase outcomes (integer values in binary format q2 q1 q0):")
print(counts_shor)

# The measured integer yields phase phi = integer / 8
# The period r satisfies phi = k / r. 
for outcome, occurrences in counts_shor.items():
    val = int(outcome, 2)
    phase = val / 8
    print(f"Measured: {outcome} (decimal {val}) | Phase: {phase:.3f} | Occurrences: {occurrences}")


Measured Phase outcomes (integer values in binary format q2 q1 q0):
{'000': 244, '100': 220, '010': 134, '011': 133, '101': 43, '111': 191, '110': 35}
Measured: 000 (decimal 0) | Phase: 0.000 | Occurrences: 244
Measured: 100 (decimal 4) | Phase: 0.500 | Occurrences: 220
Measured: 010 (decimal 2) | Phase: 0.250 | Occurrences: 134
Measured: 011 (decimal 3) | Phase: 0.375 | Occurrences: 133
Measured: 101 (decimal 5) | Phase: 0.625 | Occurrences: 43
Measured: 111 (decimal 7) | Phase: 0.875 | Occurrences: 191
Measured: 110 (decimal 6) | Phase: 0.750 | Occurrences: 35
